Paper panel: cell boundaries in one cortex sample, colored by Glut / NeuN-low Glut / ECs proximal or distal to NeuN-low neurons, with the EC distance cutoff drawn as circles. Input: EC-proximity object from 04. Needs the `sdata` env.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import spatialdata as sd
import spatialdata_plot  # noqa: F401
from matplotlib.patches import Circle, Rectangle
from matplotlib_scalebar.scalebar import ScaleBar
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["font.family"] = "Arial" if "Arial" in [f.name for f in fm.fontManager.ttflist] else "sans-serif"

In [ ]:
sample = "foxf2_s1_r0"
radius = 18  # um, one of the cutoffs 18/31/58/103 (04)
size = 200  # zoom box width, um

In [ ]:
base = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark")
method_path = base / "analysis/foxf2/Cellpose_1_nuclei_model"
shapes_key = "boundaries_Cellpose_1_nuclei_model"

In [ ]:
adata = sc.read_h5ad(method_path / "adatas/20261018_temp-after_ec_proximal_to_lowNeuN.h5ad.gz")
adata = adata[adata.obs["sample"] == sample].copy()

palette = {"Neurons-Glut": "#e5e5e5", "Neurons-Glut-lowNeuN": "#b2182b", "ECs-proximal": "#08306b", "ECs-distal": "#6baed6"}
label = adata.obs["cell_type_incl_lowNeuN"].astype(str)
ec = adata.obs["cell_type"] == "ECs"
label[ec] = adata.obs.loc[ec, f"ec_proximal_to_lowNeuN_{radius}um"].map({True: "ECs-proximal", False: "ECs-distal"})
adata.obs["label"] = pd.Categorical(label, categories=list(palette))
adata.obs["region"] = shapes_key
adata.obs["cell_id"] = adata.obs_names

# zoom box centered on the cortex spot with the most proximal ECs
xy = adata.obsm["spatial_microns"][:, :2][((adata.obs["label"] == "ECs-proximal") & (adata.obs["brain_region"] == "CTX")).to_numpy()]
cx, cy = xy[cKDTree(xy).query_ball_point(xy, size / 2, return_length=True).argmax()]
xmin, ymin, xmax, ymax = cx - size / 2, cy - size / 2, cx + size / 2, cy + size / 2
adata.obs["label"].value_counts()

In [ ]:
sdata = sd.SpatialData(
    shapes={shapes_key: sd.read_zarr(base / "samples" / sample / "sdata_z3.zarr")[shapes_key]},
    tables={"table": sd.models.TableModel.parse(adata, region=shapes_key, region_key="region", instance_key="cell_id")},
)
render = dict(element=shapes_key, color="label", groups=list(palette), palette=list(palette.values()), table_name="table")

In [ ]:
sdata.pl.render_shapes(**render).pl.show(coordinate_systems="micron", title=f"{sample}, EC radius {radius} µm")
ax = plt.gca()
ax.add_patch(Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, edgecolor="grey", linewidth=0.5))
ax.add_artist(ScaleBar(dx=1, units="µm", fixed_value=500, location="lower left", height_fraction=0.007))
for ext in ("png", "svg"):
    plt.savefig(method_path / f"neuronal-subclustering/overview_boundary_box_{sample}_EC-radius-{radius}um_spatial.{ext}", dpi=800, bbox_inches="tight")
plt.show()

In [ ]:
sub = sdata.query.bounding_box(axes=["x", "y"], min_coordinate=[xmin, ymin], max_coordinate=[xmax, ymax], target_coordinate_system="micron")
sub.pl.render_shapes(**render).pl.show(coordinate_systems="micron", title=f"EC radius: {radius} µm", figsize=(8, 8), colorbar=False)
ax = plt.gca()
for x, y in sub["table"][sub["table"].obs["label"].str.startswith("ECs")].obsm["spatial_microns"][:, :2]:
    ax.add_patch(Circle((x, y), radius, edgecolor="grey", facecolor="none", linewidth=0.5, alpha=0.8))
ax.add_artist(ScaleBar(dx=1, units="µm", fixed_value=50, location="lower left"))
ax.set(xticks=[], yticks=[], xlabel="", ylabel="")
for ext in ("png", "svg"):
    plt.savefig(method_path / f"neuronal-subclustering/boundary_box_{sample}_EC_radius_{radius}um_spatial.{ext}", dpi=800, bbox_inches="tight")
plt.show()